# Notebook 04 — Explicabilidad SHAP
**Summary · Importancia · Waterfall · Dependence**

In [ ]:
!pip install shap xgboost scikit-learn matplotlib -q
print('✓ Listo')

In [ ]:
import pandas as pd, numpy as np, pickle, warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import shap
warnings.filterwarnings('ignore')
C1='#E63946'; C2='#457B9D'; C3='#2D6A4F'; C4='#F4A261'
features = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
            'Insulin','BMI','DiabetesPedigreeFunction','Age']

from google.colab import files
uploaded = files.upload()  # sube xgboost_diabetes.pkl y diabetes_procesado.csv
import io

model_bytes = None; df = None
for nombre, contenido in uploaded.items():
    if nombre.endswith('.pkl'):
        bundle = pickle.loads(contenido)
        model=bundle['model']; scaler=bundle['scaler']
    elif nombre.endswith('.csv'):
        df = pd.read_csv(io.BytesIO(contenido))

from sklearn.model_selection import train_test_split
X = df[features].values; y = df['Outcome'].values
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
X_test_sc = scaler.transform(X_test)
X_test_df = pd.DataFrame(X_test_sc, columns=features)

explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_df)
mean_shap   = np.abs(shap_values).mean(axis=0)
print(f'✓ SHAP calculado: {shap_values.shape}')
print(f'\nTop 3 variables:')
for i in np.argsort(mean_shap)[::-1][:3]:
    print(f'  {features[i]}: {mean_shap[i]:.4f}')

In [ ]:
# Figura 8 — SHAP Summary Plot
fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_df, feature_names=features, show=False)
plt.title('SHAP Summary Plot — Importancia y dirección de variables\nXGBoost · Predicción de riesgo de diabetes',
          fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('08_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show(); print('✓ Figura 8 guardada')

In [ ]:
# Figura 9 — SHAP Importancia global
idx_sorted = np.argsort(mean_shap)
feat_sorted = [features[i] for i in idx_sorted]
shap_sorted = mean_shap[idx_sorted]
fig, ax = plt.subplots(figsize=(10, 6))
cols_bar = [C1 if v==max(mean_shap) else C2 for v in shap_sorted]
bars = ax.barh(feat_sorted, shap_sorted, color=cols_bar, alpha=0.85, height=0.6)
for bar, val in zip(bars, shap_sorted):
    ax.text(bar.get_width()+0.002, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Importancia global de variables — SHAP', fontsize=12, fontweight='bold')
ax.axvline(mean_shap.mean(), color='gray', linestyle='--', lw=1.2, label=f'Media: {mean_shap.mean():.4f}')
ax.legend(fontsize=9); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('09_shap_importancia.png', dpi=150, bbox_inches='tight')
plt.show(); print('✓ Figura 9 guardada')

In [ ]:
# Figura 10 — Waterfall 3 pacientes
y_pred_proba = model.predict_proba(X_test_df)[:,1]
idx_alto = np.where(y_pred_proba > 0.75)[0]
idx_bajo = np.where(y_pred_proba < 0.25)[0]
idx_lim  = np.where((y_pred_proba>0.45)&(y_pred_proba<0.55))[0]
casos = []
if len(idx_alto): casos.append((idx_alto[0],'Alto riesgo',C1,y_pred_proba[idx_alto[0]]))
if len(idx_bajo): casos.append((idx_bajo[0],'Bajo riesgo',C3,y_pred_proba[idx_bajo[0]]))
if len(idx_lim):  casos.append((idx_lim[0], 'Caso límite',C4,y_pred_proba[idx_lim[0]]))
patch_pos = mpatches.Patch(color=C1, label='↑ Aumenta riesgo')
patch_neg = mpatches.Patch(color=C3, label='↓ Reduce riesgo')
fig, axes = plt.subplots(1,len(casos),figsize=(6*len(casos),7))
if len(casos)==1: axes=[axes]
for ax,(idx,titulo,color,prob) in zip(axes,casos):
    shap_ind = shap_values[idx]
    feat_vals = X_test_df.iloc[idx]
    idx_s = np.argsort(np.abs(shap_ind))
    feat_s = [features[i] for i in idx_s]
    shap_s = shap_ind[idx_s]
    ax.barh(feat_s, shap_s, color=[C1 if v>0 else C3 for v in shap_s], alpha=0.85, height=0.6)
    ax.axvline(0,color='black',lw=1)
    for i,(f,sv) in enumerate(zip(feat_s,shap_s)):
        offset=0.003 if sv>=0 else -0.003
        ax.text(sv+offset,i,f'{feat_vals[f]:.2f}',va='center',
                ha='left' if sv>=0 else 'right',fontsize=8)
    ax.set_title(f'{titulo}\nProb. diabetes: {prob:.1%}',fontsize=10,fontweight='bold',color=color)
    ax.set_xlabel('SHAP value')
    ax.legend(handles=[patch_pos,patch_neg],fontsize=7); ax.grid(axis='x',alpha=0.3)
plt.suptitle('SHAP Waterfall — Explicación individual por paciente',fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig('10_shap_waterfall.png',dpi=150,bbox_inches='tight')
plt.show(); print('✓ Figura 10 guardada')

In [ ]:
# Figura 11 — Dependence plots
fig, axes = plt.subplots(1,2,figsize=(14,6))
for ax,feat_name in zip(axes,['Glucose','BMI']):
    feat_idx = features.index(feat_name)
    x_vals = X_test_df[feat_name].values
    s_vals = shap_values[:,feat_idx]
    sc = ax.scatter(x_vals,s_vals,c=X_test_df['BMI'].values,
                    cmap='RdYlGn_r',alpha=0.7,s=40)
    plt.colorbar(sc,ax=ax,label='BMI')
    ax.axhline(0,color='black',lw=1,linestyle='--')
    z=np.polyfit(x_vals,s_vals,1); p=np.poly1d(z)
    x_line=np.linspace(x_vals.min(),x_vals.max(),100)
    ax.plot(x_line,p(x_line),color='gray',lw=1.5,linestyle='--',label='Tendencia')
    ax.set_xlabel(f'{feat_name} (escalado)'); ax.set_ylabel(f'SHAP — {feat_name}')
    ax.set_title(f'Dependence Plot — {feat_name}',fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle('SHAP Dependence Plots — Glucosa y BMI',fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig('11_shap_dependence.png',dpi=150,bbox_inches='tight')
plt.show(); print('✓ Figura 11 guardada')

from google.colab import files
for f in ['08_shap_summary.png','09_shap_importancia.png','10_shap_waterfall.png','11_shap_dependence.png']:
    files.download(f)
print('✓ Todas las figuras SHAP descargadas')